In [1]:
import pandas as pd

In [2]:
post_edit_file = "/storage/brno2/home/rahmang/xcomet/arafat_comet/COMET_GR/postedition_aligned.final.translator.tsv"

In [3]:
# Read TSV file into DataFrame
df = pd.read_csv(post_edit_file, sep='\t')

# Display the first 5 rows
print(df.head(10))


    id_hal  Translation_id  line_id  \
0  3977982           18634        0   
1  3977982           18634        1   
2  3977982           18634        2   
3  3977982           18634        3   
4  3977982           18634        4   
5  3977982           18634        5   
6  3866253            4214        0   
7  3866253            4214        1   
8  3866253            4214        2   
9  3866253            4214        3   

                                              source  \
0  Tackling Ambiguity with Images: Improved Multi...   
1  One of the major challenges of machine transla...   
2  However, recent work in multimodal MT (MMT) ha...   
3  We present a new MMT approach based on a stron...   
4  We also release CoMMuTE, a Contrastive Multili...   
5  Our approach obtains competitive results over ...   
6          Conceptual Similarity for Subjective Tags   
7  Tagging in the context of online resources is ...   
8  Tags assist with the indexing, management, and...   
9  Traditi

In [4]:
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/XCOMET-XL")
#model = load_from_checkpoint(model_path)

/storage/brno2/home/rahmang/envs/xcomet/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 161.29it/s]


In [5]:
from comet import download_model, load_from_checkpoint
from comet.models.multitask.unified_metric import UnifiedMetric
class CustomXCOMET(UnifiedMetric):
    print("custom unified_metric")
    
# Load checkpoint into your custom class
#path = "/storage/brno2/home/rahmang/xcomet/downloadedxcomet/models--Unbabel--XCOMET-XL/snapshots/50d428488e021205a775d5fab7aacd9502b58e64/checkpoints/model.ckpt"

model = CustomXCOMET.load_from_checkpoint(model_path,strict = False)

custom unified_metric


Encoder model frozen.
/storage/brno2/home/rahmang/envs/xcomet/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


### tokenize the source from aligned translator set using xcomet's tokenizer

In [6]:
# load the tokenizer
tokenizer = model.encoder.tokenizer


def tokenize_with_offsets(text):
    """Tokenize text and return (words, offsets)"""
    tokenized = tokenizer(
        text,
        return_offsets_mapping=True,
        return_tensors="pt",
        truncation=True
    )
    
    word_ids = tokenized.word_ids()
    offset_mapping = tokenized["offset_mapping"][0].tolist()
    
    word_to_data = {}
    for idx, (word_id, (start, end)) in enumerate(zip(word_ids, offset_mapping)):
        if word_id is None:
            continue
        if word_id not in word_to_data:
            word_to_data[word_id] = {
                "tokens": [],
                "start": start,
                "end": end
            }
        else:
            word_to_data[word_id]["end"] = end
        token = tokenizer.convert_ids_to_tokens(tokenized["input_ids"][0][idx].item())
        word_to_data[word_id]["tokens"].append(token)
    
    word_mapping = []
    word_offsets = []
    for word_id in sorted(word_to_data.keys()):
        data = word_to_data[word_id]
        word = tokenizer.convert_tokens_to_string(data["tokens"]).strip()
        word_mapping.append(word)
        word_offsets.append((data["start"], data["end"]))
    
    return word_mapping, word_offsets

# Process DataFrame
def process_row(row):
    # Process translation
    source_text = row["source"][0] if isinstance(row["source"], list) else row["source"]
    source_tokens, source_offsets = tokenize_with_offsets(source_text)
    
    
    return pd.Series({
        "source_tokenized": source_tokens,
        "source_word_offset": source_offsets,
    })

# Apply to DataFrame
new_cols = df.apply(process_row, axis=1)
df = pd.concat([df, new_cols], axis=1)



Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [10]:
df.head()

,id_hal,Translation_id,line_id,source,translation,postedition,source_tokenized,source_word_offset
0,3977982,18634,0,Tackling Ambiguity with Images: Improved Multi...,S'attaquer à l'ambiguïté avec des images : Tra...,S'attaquer à l'ambiguïté avec des images : Tra...,"[Tackling, Ambiguity, with, Images:, Improved,...","[(0, 8), (8, 18), (18, 23), (23, 31), (31, 40)..."
1,3977982,18634,1,One of the major challenges of machine transla...,L'un des principaux défis de la traduction aut...,L'un des principaux défis de la traduction aut...,"[One, of, the, major, challenges, of, machine,...","[(0, 3), (3, 6), (6, 10), (10, 16), (16, 27), ..."
2,3977982,18634,2,"However, recent work in multimodal MT (MMT) ha...","Cependant, des travaux récents dans le domaine...","Cependant, des travaux récents dans le domaine...","[However,, recent, work, in, multimodal, MT, (...","[(0, 8), (8, 15), (15, 20), (20, 23), (23, 34)..."
3,3977982,18634,3,We present a new MMT approach based on a stron...,Nous présentons une nouvelle approche de la MT...,Nous présentons une nouvelle approche de la MT...,"[We, present, a, new, MMT, approach, based, on...","[(0, 2), (2, 10), (10, 12), (12, 16), (16, 20)..."
4,3977982,18634,4,"We also release CoMMuTE, a Contrastive Multili...","Nous publions également CoMMuTE, un ensemble d...","Nous publions également CoMMuTE, un ensemble d...","[We, also, release, CoMMuTE,, a, Contrastive, ...","[(0, 2), (2, 7), (7, 15), (15, 24), (24, 26), ..."


In [8]:
# Show result
df.iloc[0]["source_tokenized"]

['Tackling',
 'Ambiguity',
 'with',
 'Images:',
 'Improved',
 'Multimodal',
 'Machine',
 'Translation',
 'and',
 'Contrastive',
 'Evaluation']

In [9]:
df.iloc[1]["source_tokenized"]

['One',
 'of',
 'the',
 'major',
 'challenges',
 'of',
 'machine',
 'translation',
 '(MT)',
 'is',
 'ambiguity,',
 'which',
 'can',
 'in',
 'some',
 'cases',
 'be',
 'resolved',
 'by',
 'accompanying',
 'context',
 'such',
 'as',
 'an',
 'image.']

In [11]:
df.iloc[0]["source_word_offset"]

[(0, 8),
 (8, 18),
 (18, 23),
 (23, 31),
 (31, 40),
 (40, 51),
 (51, 59),
 (59, 71),
 (71, 75),
 (75, 87),
 (87, 98)]

In [12]:
first_source = df.iloc[0]["source"]

In [13]:
first_source

'Tackling Ambiguity with Images: Improved Multimodal Machine Translation and Contrastive Evaluation'

In [14]:
type(first_source)

str

In [15]:
first_source[0:8]

'Tackling'

In [ ]:
first_source[0:8]

In [16]:
first_source[87: 98]

' Evaluation'

In [17]:
first_source[75: 87]

' Contrastive'

In [18]:
df.to_csv("postedition_aligned_with_tokenized_offsets_translator_source_sentences.csv", index=False)
